## 1장 1강 : LLM 애플리케이션의 입력과 출력 구조

### 의존 패키지 설치
```
uv add ipykernel langchain langchain-core langchain-ollama python-dotenv
```

### 3. LCEL 파이프라인 실습

#### 3.1 환경 변수 로드 및 패키지 불러오기

In [36]:
from langchain_ollama import ChatOllama

from langchain_core.prompts import ChatPromptTemplate

#### 3.2 ChatPromptTemplate으로 입력 템플릿 조립하기

시스템 역할과 사용자 질문 변수({user_question})를 담은 템플릿 생성<br>
system: "당신은 컴퓨터 기초 개념을 일상적인 사물에 빗대어 설명하는 교육 전문가입니다. 2문장 이내로 친절하게 설명하세요."<br>
user: "{user_question}"

In [ ]:
# system: 역할, 상황, 제한, 조건, 예시 → 시스템 메세지, SystemMessage(..)
# user: 사용자의 질의: HumanMessage()
# assistant: AI의 답변: AIMessage()

prompt_template = ChatPromptTemplate.from_messages([

("system", "당신은 컴퓨터 기초 개념을 일상적인 사물에 빗대어 설명하는 교육 전문가 입니다. 2문장 이내로 친절하게 설명하세요."),
("user", "{user_question} {product}")

])

템플릿에 텍스트용 질문 데이터를 주입하여 결과를 확인

user_question: "프로그래밍에서 '변수'가 무엇인가요?"

In [41]:
from langchain_core.prompts import ChatPromptTemplate

sample_prompt = prompt_template.invoke({"user_question": "프로그래밍에서 '변수'가 무엇인가요?"})
sample_prompt

ChatPromptValue(messages=[SystemMessage(content='당신은 컴퓨터 기초 개념을 일상적인 사물에 빗대어 설명하는 교육 전문가 입니다. 2문장 이내로 친절하게 설명하세요.', additional_kwargs={}, response_metadata={}), HumanMessage(content="프로그래밍에서 '변수'가 무엇인가요?", additional_kwargs={}, response_metadata={})])

#### 3.3 ChatOllama로 mistral 모델 호출하기

ChatOllama 모델 인스턴스 생성

In [42]:
model = ChatOllama(

model = "mistral"

#base_url = “http://localhost:11434” #ollama 서버 주소

)

model

ChatOllama(metadata={'lc_versions': {'langchain-core': '1.6.2', 'langchain': '1.4.0'}}, model='mistral')

앞서 만든 sample_prompt를 모델에 직접 전달하여 실행

In [43]:
response = model.invoke(sample_prompt)

response

AIMessage(content=' 변수는 프로그래밍 언어에서 데이터를 저장하기 위한 상자와 같습니다. 이름을 지어진 상자에 데이터를 넣고, 필요한 시점에 데이터를 꺼내와 사용할 수 있습니다. 예를 들어, 가방에 사과를 담고 나면, 사과를 필요로 할 때 가방에서 빼서 사용합니다. 이와 같이 변수를 사용하면 프로그램에서 여러 번 사용할 수 있는 데이터를 관리하기 쉬워집니다.', additional_kwargs={}, response_metadata={'model': 'mistral', 'created_at': '2026-09-09T06:20:58.2623613Z', 'done': True, 'done_reason': 'stop', 'total_duration': 46046694400, 'load_duration': 25142237600, 'prompt_eval_count': 107, 'prompt_eval_duration': 17069436000, 'eval_count': 199, 'eval_duration': 3821520000, 'logprobs': None, 'model_name': 'mistral', 'model_provider': 'ollama'}, id='lc_run--01a084d2-f233-7e01-b2cb-344a96203979-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 107, 'output_tokens': 199, 'total_tokens': 306})

#### 3.4 StrOutputParser로 순수 텍스트만 추출하기

문자열 출력 파서 생성

In [44]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

raw_response 객체에서 순수 텍스트만 추출

In [45]:
text = parser.invoke(response)

text

' 변수는 프로그래밍 언어에서 데이터를 저장하기 위한 상자와 같습니다. 이름을 지어진 상자에 데이터를 넣고, 필요한 시점에 데이터를 꺼내와 사용할 수 있습니다. 예를 들어, 가방에 사과를 담고 나면, 사과를 필요로 할 때 가방에서 빼서 사용합니다. 이와 같이 변수를 사용하면 프로그램에서 여러 번 사용할 수 있는 데이터를 관리하기 쉬워집니다.'

#### 3.5 LCEL 파이프(|) 연산자로 완전한 체인 결합 및 실행하기

파이프(|) 연산자를 사용해 3개 컴포넌트를 하나의 파이프라인으로 연결

In [46]:
chain = prompt_template | model | parser

새로운 질문으로 파이프라인 전체 실행

In [48]:
res = chain.invoke(
    {
        "user_question": "클래스(class)에 대해서 알기 쉬운 비유를 통해 설명해보세요",
        "produdct": "제품"
    }
)
res

' 클래스(Class)는 실생활에서 가장자리를 놓치고 있는 아이들의 특징을 설명하는 명사입니다. 가장자리를 놓치고 있는 아이들이 어떤 특징을 가지고 있나요? 그리고 또 다른 가장자리를 놓치고 있는 아이들도 그렇게 어떤 특징이 있나요? 클래스는 공통된 특징을 가진 여러 개의 객체(가장자리를 놓치고 있는 아이들)를 설명하는 구조입니다. 예를 들어, 가장자리를 놓치고 있는 아이들은 모두 유년기 동안에 자라서 같은 학교를 다니고 있습니다. 따라서, 같은 학교 다니는 어린이들을 설명하려면 "학교 다니는 아이"라는 명사만 사용하면 되지 않나요? 하지만 학교 다니는 아이들이 다양한 특징을 가지고 있기 때문에, 그들의 공통된 특징에 대한 설명이 필요합니다. 이런 공통된 특징을 정의하는 것이 클래스의 역할입니다.'

### 4. 프롬프트 엔지니어링
#### 4.1 프롬프트 엔지니어링의 3대 핵심 역할
- 목표 명확화
- 제약 조건 부여
- 출력 규격 표준화

#### 4.2 AI 성능 확장 4단계 비교
- Prompt Engineering
- RAG (검색 증강 생성)
- Fine-tuning (미세 조정)
- AI Agent (에이전트)